In [8]:
import pandas as pd
import numpy as np

In [7]:
# ======================================================
# Parameter counters
# ======================================================

def mlp_parameters(
    input_dim,
    hidden_width,
    hidden_layers,
    output_dim=1,
):
    dims = [input_dim] + [hidden_width] * hidden_layers + [output_dim]

    total = 0
    for din, dout in zip(dims[:-1], dims[1:]):
        total += din * dout + dout

    return total


def kan_parameters(
    input_dim,
    hidden_width,
    hidden_layers,
    grid=3,
    spline_order=3,
    output_dim=1,
):
    dims = [input_dim] + [hidden_width] * hidden_layers + [output_dim]

    total = 0

    factor = grid + spline_order + 3

    for din, dout in zip(dims[:-1], dims[1:]):
        total += (din * dout) * factor + dout

    return total

In [ ]:
# ======================================================
# Search space
# ======================================================

mlp_widths = [75, 100, 125]
mlp_depths = [3, 4, 5]

kan_widths = np.arange(10, 100, 1)
kan_depths = [3, 4, 5]

GRID = 3
ORDER = 3

# ======================================================
# Build tables
# ======================================================

rows = []

for depth in mlp_depths:

    for w_mlp in mlp_widths:

        p_mlp = 2 * mlp_parameters(
            input_dim=2,
            hidden_width=w_mlp,
            hidden_layers=depth,
        )

        for w_kan in kan_widths:

            p_kan = 2 * kan_parameters(
                input_dim=2,
                hidden_width=w_kan,
                hidden_layers=depth,
                grid=GRID,
                spline_order=ORDER,
            )

            diff = abs(p_mlp - p_kan)

            rows.append(
                {
                    "Layers": depth,
                    "MLP width": w_mlp,
                    "MLP params": p_mlp,
                    "KAN width": w_kan,
                    "KAN params": p_kan,
                    "Difference": diff,
                }
            )

df = pd.DataFrame(rows)

# Keep only the closest KAN for every MLP architecture
best = (
    df.sort_values("Difference")
      .groupby(["Layers", "MLP width"], as_index=False)
      .first()
)

In [10]:
best

,Layers,MLP width,MLP params,KAN width,KAN params,Difference
0,3,75,23402,25,24002,600
1,3,100,41202,33,41186,16
2,3,125,64002,41,62978,1024
3,4,75,34802,25,35302,500
4,4,100,61402,33,60854,548
5,4,125,95502,41,93318,2184
6,5,75,46202,25,46602,400
7,5,100,81602,33,80522,1080
8,5,125,127002,42,129698,2696
